<img src="https://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

# Deep Learning Basics with PyTorch

**Dr. Yves J. Hilpisch with GPT-5**

# Part IV — Toward Large Language Models

## Chapter 15 — Exercises: Training Large Models

## Overview

Three exercises that extend the Chapter 15 notebook.  Each exercise can be
completed on **CPU** — no GPU is required.

| # | Exercise | Core concept |
|---|----------|--------------|
| 1 | **Accumulation sweep + gradient equivalence** | `B_global`, wall-clock, numerical verification |
| 2 | **Checkpoint interrupt and resume** | Save after N epochs, reload, verify loss continuity |
| 3 | **GradScaler overflow simulation** | Force overflow, observe skip + scale decay, plot scale history |

Work through each exercise in order; later exercises build on state set up by earlier ones.
Run cells top-to-bottom and restart the kernel between independent attempts.

## Shared Setup

Imports, synthetic dataset, and `make_model` helper reused across all exercises.

In [ ]:
import math
import sys
import time
import tempfile
import pathlib

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print(f'Python : {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

torch.manual_seed(42)
N, D = 1024, 32
X = torch.randn(N, D)
y = torch.randn(N, 1)
dataset = TensorDataset(X, y)


def make_model(hidden: int = 64) -> nn.Module:
    """Fresh TinyMLP — same pattern as Ch 8/12."""
    return nn.Sequential(
        nn.Linear(D, hidden),
        nn.ReLU(),
        nn.Linear(hidden, 1),
    ).to(device)


print(f'Dataset : {N} × {D}   Model: {D}→{64}→1')

---
## Exercise 1 — Accumulation Sweep and Gradient Equivalence

**Goal:** understand the cost/benefit trade-off of gradient accumulation and
confirm that K micro-batches produce the same weight update as one large batch.

### Tasks

1. Sweep `K ∈ {1, 2, 4, 8, 16}` with a fixed micro-batch size of 16.
2. For each K, run one epoch and record:
   - `B_global = B_micro × K` (effective batch size)
   - wall-clock time (ms)
   - final step loss
3. Plot **time vs K** and **B_global vs K** side-by-side.
4. For K=4, numerically verify that the accumulated update matches a single
   batch of size `B_micro × K` (max parameter delta < 1e-5).

**Expected insight:** time grows roughly linearly with K on CPU (more backward
calls); on a memory-bound GPU it stays nearly flat until compute dominates.

In [ ]:
# --- Task 1-3: sweep K and record metrics ---------------------------
B_MICRO      = 16
K_VALUES     = [1, 2, 4, 8, 16]

records = []   # list of (K, B_global, elapsed_ms, final_loss)

for K in K_VALUES:
    m  = make_model()
    o  = torch.optim.AdamW(m.parameters(), lr=1e-3)
    ld = DataLoader(dataset, batch_size=B_MICRO, shuffle=False)
    m.train()

    t0         = time.perf_counter()
    final_loss = None
    o.zero_grad(set_to_none=True)

    for step, (xb, yb) in enumerate(ld):
        xb, yb = xb.to(device), yb.to(device)
        # ── YOUR CODE: forward + scaled backward + conditional step ──
        loss = F.mse_loss(m(xb), yb) / K          # scale loss by K
        loss.backward()
        if (step + 1) % K == 0:
            o.step()
            o.zero_grad(set_to_none=True)
        final_loss = loss.item() * K              # unscale for logging

    elapsed_ms = (time.perf_counter() - t0) * 1000
    B_global   = B_MICRO * K
    records.append((K, B_global, elapsed_ms, final_loss))
    print(f'K={K:2d}  B_global={B_global:4d}  '
          f'time={elapsed_ms:6.1f} ms  loss={final_loss:.4f}')

# --- Plot -----------------------------------------------------------
ks      = [r[0] for r in records]
times   = [r[2] for r in records]
batches = [r[1] for r in records]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3))
ax1.plot(ks, times, marker='o', color='steelblue')
ax1.set_xlabel('Accumulation steps K')
ax1.set_ylabel('Time per epoch (ms)')
ax1.set_title('Wall-clock vs K')
ax1.set_xticks(ks)

ax2.bar(ks, batches, color='salmon', width=0.6)
ax2.set_xlabel('Accumulation steps K')
ax2.set_ylabel('Effective batch size')
ax2.set_title('B_global vs K')
ax2.set_xticks(ks)

fig.tight_layout()
plt.show()

In [ ]:
# --- Task 4: verify gradient equivalence for K=4 --------------------
K_CHECK = 4
B_CHECK = B_MICRO * K_CHECK   # 64

torch.manual_seed(7)
big_X = torch.randn(B_CHECK, D, device=device)
big_y = torch.randn(B_CHECK, 1, device=device)

# Reference: one large batch
torch.manual_seed(99)
ref = make_model()
ref_o = torch.optim.SGD(ref.parameters(), lr=0.1)
ref_o.zero_grad(set_to_none=True)
F.mse_loss(ref(big_X), big_y).backward()
ref_o.step()

# Accumulation: K_CHECK micro-batches
torch.manual_seed(99)          # same init weights
acc = make_model()
acc_o = torch.optim.SGD(acc.parameters(), lr=0.1)
acc_o.zero_grad(set_to_none=True)
for cx, cy in zip(big_X.chunk(K_CHECK), big_y.chunk(K_CHECK)):
    # ── YOUR CODE: scaled backward for each chunk ─────────────────
    (F.mse_loss(acc(cx), cy) / K_CHECK).backward()
acc_o.step()

max_delta = max(
    (acc.state_dict()[k] - ref.state_dict()[k]).abs().max().item()
    for k in acc.state_dict()
)
status = 'PASS ✓' if max_delta < 1e-5 else f'FAIL — delta={max_delta:.2e}'
print(f'Gradient equivalence (K={K_CHECK}): max_delta={max_delta:.2e}  [{status}]')

### Reflection

Answer in the cell below (edit as markdown):

1. What happens to wall-clock time as K doubles on CPU?  Does this match your expectation?
2. Why is dividing the loss by K — and not the gradients directly — the correct approach?
3. In what scenario would you choose K=8 over K=2?

**Your answers here:**

1. ...
2. ...
3. ...

---
## Exercise 2 — Checkpoint Interrupt and Resume

**Goal:** simulate a training crash at epoch 3 out of 6, reload the checkpoint,
continue training to epoch 6, and verify the final loss matches a full uninterrupted run.

### Tasks

1. Train a model for **6 epochs** (`full_run`), recording loss per epoch.
2. Train a second model for **3 epochs** (`interrupted_run`), save a checkpoint,
   then reload it and continue for 3 more epochs (`resumed_run`).
3. Compare `full_run` and `resumed_run` final loss — they should match
   because both start from the same seed and use the same data order
   (no shuffle, same `torch.manual_seed`).
4. Plot loss curves for all three runs on the same axes.

**Note:** perfect numerical match requires identical RNG state at resume point,
which we enforce by saving and restoring `torch.get_rng_state()`.

In [ ]:
EPOCHS      = 6
CRASH_EPOCH = 3    # save checkpoint after this epoch
LR          = 1e-2
LOADER_BS   = 32
SEED        = 0

ckpt_dir = pathlib.Path(tempfile.mkdtemp())

# ── Helper: one epoch ────────────────────────────────────────────────
def run_epoch(model, opt, loader):
    model.train()
    total = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(set_to_none=True)
        loss = F.mse_loss(model(xb), yb)
        loss.backward()
        opt.step()
        total += loss.item() * xb.size(0)
    return total / len(loader.dataset)

# ── Full run (no interruption) ───────────────────────────────────────
torch.manual_seed(SEED)
full_model  = make_model()
full_opt    = torch.optim.AdamW(full_model.parameters(), lr=LR)
full_loader = DataLoader(dataset, batch_size=LOADER_BS, shuffle=False)
full_losses = []

for ep in range(1, EPOCHS + 1):
    l = run_epoch(full_model, full_opt, full_loader)
    full_losses.append(l)
    print(f'[full]  epoch {ep:2d}  loss={l:.5f}')

In [ ]:
# ── Interrupted run: train → crash → save checkpoint ────────────────
torch.manual_seed(SEED)          # same init as full run
int_model  = make_model()
int_opt    = torch.optim.AdamW(int_model.parameters(), lr=LR)
int_sched  = torch.optim.lr_scheduler.CosineAnnealingLR(int_opt, T_max=EPOCHS)
int_loader = DataLoader(dataset, batch_size=LOADER_BS, shuffle=False)
int_losses = []

for ep in range(1, CRASH_EPOCH + 1):
    l = run_epoch(int_model, int_opt, int_loader)
    int_sched.step()
    int_losses.append(l)
    print(f'[interrupted]  epoch {ep:2d}  loss={l:.5f}')

# ── Save checkpoint ─────────────────────────────────────────────────
ckpt_path = ckpt_dir / 'ex2_interrupt.pt'
torch.save({
    'model' : int_model.state_dict(),
    'opt'   : int_opt.state_dict(),
    'sched' : int_sched.state_dict(),
    'epoch' : CRASH_EPOCH,
    'rng'   : torch.get_rng_state(),
}, ckpt_path)
print(f'\nCheckpoint saved → {ckpt_path}  (after epoch {CRASH_EPOCH})')

In [ ]:
# ── Resume run: reload checkpoint → continue to EPOCHS ──────────────
loaded = torch.load(ckpt_path, map_location=device, weights_only=False)

res_model  = make_model()        # fresh architecture
res_opt    = torch.optim.AdamW(res_model.parameters(), lr=LR)
res_sched  = torch.optim.lr_scheduler.CosineAnnealingLR(res_opt, T_max=EPOCHS)
res_loader = DataLoader(dataset, batch_size=LOADER_BS, shuffle=False)

# ── YOUR CODE: restore state from checkpoint ─────────────────────────
res_model.load_state_dict(loaded['model'])
res_opt.load_state_dict(loaded['opt'])
res_sched.load_state_dict(loaded['sched'])
torch.set_rng_state(loaded['rng'])       # restore RNG for reproducible data order
start_epoch = loaded['epoch']

res_losses = int_losses.copy()   # include pre-crash losses for plotting
for ep in range(start_epoch + 1, EPOCHS + 1):
    l = run_epoch(res_model, res_opt, res_loader)
    res_sched.step()
    res_losses.append(l)
    print(f'[resumed]  epoch {ep:2d}  loss={l:.5f}')

# ── Compare final losses ─────────────────────────────────────────────
delta = abs(full_losses[-1] - res_losses[-1])
print(f'\nFull final loss   : {full_losses[-1]:.6f}')
print(f'Resumed final loss: {res_losses[-1]:.6f}')
print(f'Delta             : {delta:.2e}')

In [ ]:
# ── Plot all three curves ────────────────────────────────────────────
epochs_axis = list(range(1, EPOCHS + 1))

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(epochs_axis, full_losses, label='Full run',
        marker='o', linewidth=2)
ax.plot(epochs_axis[:CRASH_EPOCH], int_losses, label='Interrupted',
        marker='s', linestyle='--', color='orange')
ax.plot(epochs_axis, res_losses, label='Resumed',
        marker='^', linestyle=':', color='green')
ax.axvline(CRASH_EPOCH + 0.5, color='red', linestyle='--',
           linewidth=1, label='Crash point')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE loss')
ax.set_title('Full vs interrupted + resumed run')
ax.legend()
fig.tight_layout()
plt.show()

### Reflection

1. Does the resumed curve merge cleanly with the full run after the crash point?
   If there is a small discrepancy, what could cause it?
2. Why is saving the scheduler state important here (not just the model and optimizer)?
3. What would you need to change for the checkpoint to work in a DDP setting?

**Your answers here:**

1. ...
2. ...
3. ...

---
## Exercise 3 — GradScaler Overflow Simulation

**Goal:** observe how `GradScaler` detects gradient overflow, skips the
optimizer step, and reduces the scale factor — without crashing training.

This exercise only runs if CUDA is available.  If you are on CPU, the
mock section below shows the same logic with manual scale tracking.

### Tasks

1. Create a `GradScaler` with an **artificially large initial scale** (`init_scale=2**24`)
   so that overflow occurs on the first step.
2. Run 20 training steps and record after each step:
   - whether the step was skipped (overflow detected)
   - the current scale value
3. Plot the scale history.
4. Confirm the model still converges after the scale stabilises.

In [ ]:
# ── CUDA path ─────────────────────────────────────────────────────────
if torch.cuda.is_available():
    STEPS      = 20
    INIT_SCALE = 2 ** 24      # deliberately too large → triggers overflow

    ovf_model  = make_model().cuda()
    ovf_opt    = torch.optim.AdamW(ovf_model.parameters(), lr=1e-3)
    ovf_loader = DataLoader(dataset, batch_size=32, shuffle=True)

    # ── YOUR CODE: create GradScaler with init_scale ──────────────────
    scaler = torch.amp.GradScaler('cuda', init_scale=INIT_SCALE)

    scale_history = []
    skipped       = []
    losses        = []

    data_iter = iter(ovf_loader)
    ovf_model.train()

    for step in range(STEPS):
        try:
            xb, yb = next(data_iter)
        except StopIteration:
            data_iter = iter(ovf_loader)
            xb, yb = next(data_iter)

        xb, yb = xb.cuda(), yb.cuda()

        ovf_opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            loss = F.mse_loss(ovf_model(xb), yb)

        scale_before = scaler.get_scale()
        scaler.scale(loss).backward()
        scaler.step(ovf_opt)
        scaler.update()
        scale_after  = scaler.get_scale()

        was_skipped = (scale_after < scale_before)  # scale dropped → overflow
        scale_history.append(scale_after)
        skipped.append(was_skipped)
        losses.append(loss.item())

        mark = ' ← SKIPPED (overflow)' if was_skipped else ''
        print(f'step {step+1:2d}  scale={scale_after:.0f}  '
              f'loss={loss.item():.4f}{mark}')

    # ── Plot ───────────────────────────────────────────────────────────
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
    ax1.semilogy(range(1, STEPS + 1), scale_history,
                 marker='o', color='steelblue')
    for s, sk in enumerate(skipped):
        if sk:
            ax1.axvline(s + 1, color='red', alpha=0.4, linewidth=1)
    ax1.set_xlabel('Step')
    ax1.set_ylabel('GradScaler scale (log)')
    ax1.set_title('Scale history (red = overflow / skip)')

    ax2.plot(range(1, STEPS + 1), losses, marker='s', color='salmon')
    ax2.set_xlabel('Step')
    ax2.set_ylabel('Loss')
    ax2.set_title('Loss (skipped steps still appear)')

    fig.tight_layout()
    plt.show()

else:
    print('CUDA not available — running CPU mock instead.')

In [ ]:
# ── CPU mock: simulate scale dynamics without CUDA ────────────────────
# GradScaler halves the scale on overflow (inf/NaN in grads).
# We reproduce the scale decay curve manually to illustrate the concept.

if not torch.cuda.is_available():
    STEPS        = 20
    init_scale   = 2 ** 24
    growth_factor = 2.0
    backoff      = 0.5
    growth_interval = 2000   # steps between scale doublings (long → mainly see backoffs)

    # Simulate: overflow on first 3 steps, then stable
    overflow_steps = {1, 2, 3}
    scale   = float(init_scale)
    history = []

    for s in range(1, STEPS + 1):
        overflowed = s in overflow_steps
        if overflowed:
            scale *= backoff          # halve on overflow
        history.append((s, scale, overflowed))
        mark = ' ← overflow (mock)' if overflowed else ''
        print(f'step {s:2d}  scale={scale:.0f}{mark}')

    fig, ax = plt.subplots(figsize=(6, 3))
    steps_  = [h[0] for h in history]
    scales_ = [h[1] for h in history]
    ovfs_   = [h[2] for h in history]

    ax.semilogy(steps_, scales_, marker='o', color='steelblue')
    for s, sk in zip(steps_, ovfs_):
        if sk:
            ax.axvline(s, color='red', alpha=0.5, linewidth=1)
    ax.set_xlabel('Step')
    ax.set_ylabel('Scale (log)')
    ax.set_title('GradScaler scale dynamics (CPU mock, red = overflow)')
    fig.tight_layout()
    plt.show()

### Reflection

1. How many steps were skipped before the scale stabilised?
2. Why does a skipped step not crash training — what does `scaler.step()` do when overflow is detected?
3. If the scale keeps decreasing and never stabilises, what does that indicate about your training setup?

**Your answers here:**

1. ...
2. ...
3. ...

---
## Summary

| Exercise | Key takeaway |
|----------|--------------|
| 1 — Accumulation sweep | K micro-batches are mathematically equivalent to one large batch when loss is scaled by 1/K; wall-clock grows with K on CPU |
| 2 — Checkpoint resume | Saving model + optimizer + scheduler + RNG state enables bit-exact resumption; scheduler state matters for LR continuity |
| 3 — Overflow simulation | GradScaler silently skips overflowed steps and halves the scale; training is safe but slower until the scale stabilises |

<img src="https://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>